In [17]:
from rdkit import Chem
from rdkit.Chem import AllChem

def smiles_to_gjf(smiles, filename="output.gjf"):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)  # Add hydrogen atoms
    AllChem.EmbedMolecule(mol)  # Generate 3D coordinates
    AllChem.UFFOptimizeMolecule(mol)  # Optimize geometry

    with open(filename, "w") as f:
        f.write("%NProcShared=4\n%Mem=4GB\n#P B3LYP/6-31G(d) Opt Freq\n\n")
        f.write(f"Generated by RDKit\n\n0 1\n")  # Charge = 0, Multiplicity = 1
        
        for atom in mol.GetAtoms():
            pos = mol.GetConformer().GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}\n")

        f.write("\n")

In [18]:
generated_molecule = "O=P(c1ccccc1)(c1ccccc1).FC(F)(F)c1cc(B(c2cc(F)c(F)c(F)c2)(c2cc(F)c(F)c(F)c2)c2cc(F)c(F)c(F)c2)cc(c1)F"

In [19]:
smiles_to_gjf(generated_molecule, "example.gjf")

[10:14:00] Explicit valence for atom # 21 B, 4, is greater than permitted


ArgumentError: Python argument types in
    rdkit.Chem.rdmolops.AddHs(NoneType)
did not match C++ signature:
    AddHs(RDKit::ROMol mol, bool explicitOnly=False, bool addCoords=False, boost::python::api::object onlyOnAtoms=None, bool addResidueInfo=False)

In [34]:
from rdkit import Chem
from rdkit.Chem import AllChem

def robust_smiles_to_gjf(smiles, filename="output.gjf"):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    # Use ETKDGv3 for embedding
    params = AllChem.ETKDGv3()
    params.randomSeed = 0xf00d
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError("❌ Embedding failed")

    # Optimize geometry
    if AllChem.UFFOptimizeMolecule(mol, maxIters=2000) != 0:
        raise RuntimeError("❌ UFF optimization failed")

    # Check distances
    conf = mol.GetConformer()
    for i in range(mol.GetNumAtoms()):
        for j in range(i+1, mol.GetNumAtoms()):
            pos_i = conf.GetAtomPosition(i)
            pos_j = conf.GetAtomPosition(j)
            dist = pos_i.Distance(pos_j)
            if dist < 0.5:
                raise ValueError(f"❌ Atoms {i} and {j} too close: {dist:.2f} Å")

    # Write GJF
    with open(filename, "w") as f:
        f.write("%NProcShared=4\n%Mem=4GB\n%Chk=output.chk\n")
        f.write("#P B3LYP/6-31G(d) SP Pop=Full\n\nGenerated from SMILES\n\n0 1\n")
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}\n")
        f.write("\n")

    print(f"✅ GJF file written: {filename}")

# Example:
robust_smiles_to_gjf("OB1OC(C)(C)C(C)(C)O1.P(C)(C)C")


✅ GJF file written: output.gjf


In [40]:
from rdkit import Chem
from rdkit.Chem import AllChem
import os

def robust_smiles_to_gjf(
    smiles: str,
    filename: str = "output.gjf",
    title: str = "Orbital-only single point calculation",
    method: str = "B3LYP",
    basis: str = "6-31G(d)",
    mem: str = "4GB",
    nproc: int = 4,
    charge: int = 0,
    multiplicity: int = 1,
    max_atoms: int = 150
):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES.")

    mol = Chem.AddHs(mol)

    # Use ETKDGv3 for better initial geometries
    params = AllChem.ETKDGv3()
    params.randomSeed = 0xF00D
    params.useSmallRingTorsions = True
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError("❌ 3D embedding failed.")

    if AllChem.UFFOptimizeMolecule(mol, maxIters=2000) != 0:
        raise RuntimeError("❌ MMFF optimization failed.")

    conf = mol.GetConformer()

    # Check for dangerously short distances
    for i in range(mol.GetNumAtoms()):
        for j in range(i + 1, mol.GetNumAtoms()):
            d = conf.GetAtomPosition(i).Distance(conf.GetAtomPosition(j))
            if d < 0.5:
                raise ValueError(f"❌ Atoms {i} and {j} too close: {d:.2f} Å")

    if mol.GetNumAtoms() > max_atoms:
        raise ValueError(f"❌ Molecule has {mol.GetNumAtoms()} atoms, exceeds limit ({max_atoms})")

    # Write .gjf file
    chk_name = os.path.splitext(filename)[0] + ".chk"
    with open(filename, "w") as f:
        f.write(f"%NProcShared={nproc}\n")
        f.write(f"%Mem={mem}\n")
        f.write(f"%Chk={chk_name}\n")
        f.write(f"#P {method}/{basis} Opt=(Cartesian,CalcFC,MaxStep=10) Freq Pop=Full IOp(3/33=1)\n\n")
        f.write(f"{title}\n\n")
        f.write(f"{charge} {multiplicity}\n")

        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}\n")

        f.write("\n")

    print(f"✅ Generated '{filename}' with {mol.GetNumAtoms()} atoms.")

# Example usage:
robust_smiles_to_gjf("B(c1c(F)ccc(F)c1)(c1c(F)ccc(F)c1)c1c(F)ccc(F)c1.Cc1cccc(C)n1", filename="test2.gjf")

ValueError: ❌ Atoms 9 and 28 too close: 0.50 Å

In [42]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdMolTransforms import ComputeCentroid

def shift_second_fragment(mol, shift_vec=(5.0, 0.0, 0.0)):
    conf = mol.GetConformer()
    frags = Chem.GetMolFrags(mol, asMols=False, sanitizeFrags=False)
    # Count atoms per fragment
    atom_to_frag = {atom_idx: frag_id for frag_id, atoms in enumerate(frags) for atom_idx in atoms}
    # Pick fragment ID of the second molecule (e.g. base)
    frag_counts = {v: list(atom_to_frag.values()).count(v) for v in set(atom_to_frag.values())}
    second_frag = sorted(frag_counts)[-1]

    # Apply shift to all atoms in that fragment
    for i in range(mol.GetNumAtoms()):
        if atom_to_frag[i] == second_frag:
            pos = conf.GetAtomPosition(i)
            new_pos = [pos.x + shift_vec[0], pos.y + shift_vec[1], pos.z + shift_vec[2]]
            conf.SetAtomPosition(i, new_pos)

def generate_safe_flp(smiles, filename="flp.gjf"):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    shift_second_fragment(mol, shift_vec=(6.0, 0.0, 0.0))
    AllChem.UFFOptimizeMolecule(mol)

    # Write GJF
    with open(filename, "w") as f:
        f.write("%NProcShared=4\n%Mem=4GB\n%Chk=flp.chk\n")
        f.write("#P B3LYP/6-31G(d) SP Pop=Full IOp(3/33=1)\n\nFLP CO2-active system\n\n0 1\n")
        conf = mol.GetConformer()
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            f.write(f"{atom.GetSymbol()} {pos.x:.6f} {pos.y:.6f} {pos.z:.6f}\n")
        f.write("\n")

generate_safe_flp("C#CC1=CN=C(O1)OO")